In [1]:
import sys

from networkx import DiGraph
from pathlib import Path

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.graph_stats as graph_stats
import utils.graph_utils as graph_utils
import utils.vis as vis

from config_handlers.dataset_handler import DatasetHandler
from config_handlers.sk_handler import SKHandler

# Main

In [2]:
sk_hdl = SKHandler()
DATASETS = sk_hdl.get_ds_list()
ds_hdl = DatasetHandler()

base_dir = Path("..")

all_datasets = DATASETS * 3
all_datasets.sort()

In [3]:
# Loading concept and predicate frequencies from VG1800 dataset.
# VG1800 is a subsample of VG made by the authors of IETrans.
vg1800_dict = graph_utils.load_vg1800_dict()

In [4]:
def get_sk_details(sks: dict) -> dict:
    data = {
        "graphs": {ds: {} for ds in DATASETS},
        "summaries": {ds: {} for ds in DATASETS},
        "sample_stats": {ds: {} for ds in DATASETS},
        "count_concepts": {ds: {} for ds in DATASETS},
        "count_preds": {ds: {} for ds in DATASETS},
    }

    for ds in sks.keys():
        print(f"Processing {ds}")
        sk_hdl.set_curr_ds(ds)

        for ds_class in sks[ds].keys():
            sk_hdl.set_curr_class(ds_class)

            # No distinction between classes, put everything from 1 dataset together
            for sk_idx, sk_data in sks[ds][ds_class].items():
                data["graphs"][ds][sk_idx] = graph_utils.sk_to_nx(sk_data)

            sample_stats, stats_summary = graph_stats.compute_stats(
                data["graphs"][ds], vg1800_dict
            )

        # Update
        data["sample_stats"][ds] = sample_stats
        data["summaries"][ds] = stats_summary
        data["count_concepts"][ds] = stats_summary["count_concepts"]
        data["count_preds"][ds] = stats_summary["count_preds"]

    return data

Compute stats for SKs pre-crowdsourcing

In [5]:
def get_sk_crowd() -> dict:
    all_sks = {}

    for ds in DATASETS:
        sk_hdl.set_curr_ds(ds)
        ds_hdl.set_curr_ds(ds)
        if ds not in all_sks:
            all_sks[ds] = {}

        ds_qs = data_io.load_json(
            Path("..").joinpath(ds_hdl.get_crowd_questions_path())
        )

        for ds_class in sk_hdl.get_classes():
            sk_hdl.set_curr_class(ds_class)
            sk_path = base_dir.joinpath(sk_hdl.get_sg_crowd_path())
            sk_data = data_io.load_jsonl(sk_path)
            ds_qs_batch = [q for q in ds_qs if q["class"] == ds_class]

            if ds_class not in all_sks[ds]:
                all_sks[ds][ds_class] = {}

            sk_dict = {}
            # Iterate over questions
            # Get relations data based on img
            # Same only what's relevant
            for q in ds_qs_batch:
                sk_dict[q["question_id"]] = next(
                    sg["rel_clusters_unique"]
                    for sg in sk_data
                    if sg["img_id"] == q["img"]
                )
            all_sks[ds][ds_class] = sk_dict

    return all_sks

In [6]:
# Load and process SK crowd
sk_crowd = get_sk_crowd()
sk_crowd = get_sk_details(sk_crowd)

Processing llava-bench
Processing mmbench
Processing seed
Processing vqav2


Compute stats for final SKs

In [7]:
def get_sk_final(manual_ann: bool = False) -> dict:
    all_sks = {}

    for ds in DATASETS:
        sk_hdl.set_curr_ds(ds)
        if ds not in all_sks:
            all_sks[ds] = {}

        for ds_class in sk_hdl.get_classes():
            sk_hdl.set_curr_class(ds_class)
            if manual_ann:
                sk_path = base_dir.joinpath(sk_hdl.get_sk_exp_path())
            else:
                sk_path = base_dir.joinpath(sk_hdl.get_sk_final_path())
            sk_data = data_io.load_jsonl(sk_path)

            if ds_class not in all_sks[ds]:
                all_sks[ds][ds_class] = {}

            sk_dict = {}
            for curr_sample in sk_data:
                question_id = curr_sample["question_id"]
                if question_id not in sk_dict:
                    sk_dict[question_id] = curr_sample["relations"]
            all_sks[ds][ds_class] = sk_dict

    return all_sks

In [8]:
# Load and process SK final
sk_final = get_sk_final()
sk_final = get_sk_details(sk_final)

Processing llava-bench
Processing mmbench
Processing seed
Processing vqav2


In [9]:
# for k, v in sk_data["summaries"]["mmbench"].items():
#     print(f"{k.upper()}: {v:.2f} -- {type(v)}")

Compute stats for SKs w/ manual annotations

In [10]:
# Load and process SK final
sk_exp = get_sk_final(manual_ann=True)
sk_exp = get_sk_details(sk_exp)

Processing llava-bench
Processing mmbench
Processing seed
Processing vqav2


Check differences at the dataset level.

In [11]:
def show_diffs(sk_crowd: dict, sk_final: dict, sk_exp: dict):
    for ds in DATASETS:
        print("=" * 25, f"{ds.upper()}", "=" * 25)
        print("\t\t = Original = \t = Final = \t = Expert =")
        avg_c_crowd = sk_crowd[ds]["avg_concepts"]
        std_c_crowd = sk_crowd[ds]["std_concepts"]
        avg_c_final = sk_final[ds]["avg_concepts"]
        std_c_final = sk_final[ds]["std_concepts"]
        avg_c_exp = sk_exp[ds]["avg_concepts"]
        std_c_exp = sk_exp[ds]["std_concepts"]
        print(
            f"Avg. Concepts \t {avg_c_crowd:.2f} \t\t {avg_c_final:.2f} \t\t {avg_c_exp:.2f}"
        )
        print(
            f"Std. Concepts \t {std_c_crowd:.2f} \t\t {std_c_final:.2f} \t\t {std_c_exp:.2f}"
        )
        print("-" * 60)
        avg_p_crowd = sk_crowd[ds]["avg_preds"]
        std_p_crowd = sk_crowd[ds]["std_preds"]
        avg_p_final = sk_final[ds]["avg_preds"]
        std_p_final = sk_final[ds]["std_preds"]
        avg_p_exp = sk_exp[ds]["avg_preds"]
        std_p_exp = sk_exp[ds]["std_preds"]
        print(
            f"Avg. Preds \t {avg_p_crowd:.2f} \t\t {avg_p_final:.2f} \t\t {avg_p_exp:.2f}"
        )
        print(
            f"Std. Preds \t {std_p_crowd:.2f} \t\t {std_p_final:.2f} \t\t {std_p_exp:.2f}"
        )
        print("-" * 60)
        avg_ic_crowd = sk_crowd[ds]["avg_ic"]
        std_ic_crowd = sk_crowd[ds]["std_ic"]
        avg_ic_final = sk_final[ds]["avg_ic"]
        std_ic_final = sk_final[ds]["std_ic"]
        avg_ic_exp = sk_exp[ds]["avg_ic"]
        std_ic_exp = sk_exp[ds]["std_ic"]
        print(
            f"Avg. IC \t {avg_ic_crowd:.2f} \t\t {avg_ic_final:.2f} \t\t {avg_ic_exp:.2f}"
        )
        print(
            f"Std. IC \t {std_ic_crowd:.2f} \t\t {std_ic_final:.2f} \t\t {std_ic_exp:.2f}"
        )
        print("-" * 60)


show_diffs(sk_crowd["summaries"], sk_final["summaries"], sk_exp["summaries"])

========================= LLAVA-BENCH =========================
		 = Original = 	 = Final = 	 = Expert =
Avg. Concepts 	 8.33 		 10.78 		 10.52
Std. Concepts 	 4.28 		 4.84 		 4.80
------------------------------------------------------------
Avg. Preds 	 9.67 		 8.60 		 8.60
Std. Preds 	 5.15 		 4.70 		 4.70
------------------------------------------------------------
Avg. IC 	 0.45 		 0.53 		 0.48
Std. IC 	 0.06 		 0.16 		 0.12
------------------------------------------------------------
========================= MMBENCH =========================
		 = Original = 	 = Final = 	 = Expert =
Avg. Concepts 	 8.85 		 8.51 		 8.47
Std. Concepts 	 2.89 		 3.20 		 3.21
------------------------------------------------------------
Avg. Preds 	 10.37 		 7.37 		 7.37
Std. Preds 	 3.79 		 3.73 		 3.73
------------------------------------------------------------
Avg. IC 	 0.47 		 0.43 		 0.41
Std. IC 	 0.06 		 0.12 		 0.11
------------------------------------------------------------
=================

Load statistics from `resolve_annotations`

In [12]:
all_count_concepts = {}
all_count_preds = {}
changes = {}

for ds in DATASETS:
    sk_hdl.set_curr_ds(ds)
    all_count_concepts[ds] = {"init": [], "val": [], "ann": []}
    all_count_preds[ds] = {"init": [], "val": [], "ann": []}
    changes[ds] = {}
    for ds_class in sk_hdl.get_classes():
        sk_hdl.set_curr_class(ds_class)

        # Loading and formatting stats
        stats_path = sk_hdl.get_sk_final_path().parent.joinpath("stats.json")
        stats_path = base_dir.joinpath(stats_path)
        stats = data_io.load_json(stats_path)
        for v in stats["samples"].values():
            # Concepts
            all_count_concepts[ds]["init"].append(v["concepts"]["init"])
            all_count_concepts[ds]["val"].append(v["concepts"]["val"])
            all_count_concepts[ds]["ann"].append(v["concepts"]["ann"])
            # Relations
            all_count_preds[ds]["init"].append(v["preds"]["init"])
            all_count_preds[ds]["val"].append(v["preds"]["val"])
            all_count_preds[ds]["ann"].append(v["preds"]["ann"])

        # Formatting summaries
        for k, v in stats["summary"]["changes"].items():
            if k not in changes[ds]:
                changes[ds][k] = 0
            changes[ds][k] += v

Check if the lower avg. concepts, predicates, and IC are tied to the number of changes (both invalid triples and updates).

In [13]:
changes

{'llava-bench': {'marked_not_relevant': 300,
  'concepts_changed': 80,
  'predicates_changed': 32,
  'new_rels': 224},
 'mmbench': {'marked_not_relevant': 839,
  'concepts_changed': 77,
  'predicates_changed': 110,
  'new_rels': 431},
 'seed': {'marked_not_relevant': 1042,
  'concepts_changed': 101,
  'predicates_changed': 126,
  'new_rels': 477},
 'vqav2': {'marked_not_relevant': 1051,
  'concepts_changed': 45,
  'predicates_changed': 100,
  'new_rels': 599}}

In [14]:
def show_changes(sk_crowd: dict, sk_final: dict, changes: dict):
    for ds in DATASETS:
        print("=" * 20, f"{ds.upper()}", "=" * 20)
        print("-" * 50)
        perc_not_relevant = (
            100
            * changes[ds]["marked_not_relevant"]
            / sk_crowd["summaries"][ds]["count_preds"]
        )
        perc_c_changed = (
            100
            * changes[ds]["concepts_changed"]
            / sk_crowd["summaries"][ds]["count_concepts"]
        )
        perc_p_changed = (
            100
            * changes[ds]["predicates_changed"]
            / sk_crowd["summaries"][ds]["count_preds"]
        )
        # Added triples / final number of triples -> fraction of "truly" new ones over the final set
        perc_new = (
            100 * changes[ds]["new_rels"] / sk_final["summaries"][ds]["count_preds"]
        )
        print(f"Initial Triples \t {sk_crowd['summaries'][ds]['count_preds']}")
        print(
            f"Triples Discarded \t {changes[ds]['marked_not_relevant']} -> {perc_not_relevant:.1f}%"
        )
        print(
            f"Final Triples \t\t {sk_final['summaries'][ds]['count_preds']} -> {perc_new:.1f}% is new"
        )
        print(
            f"Updates Concepts \t {changes[ds]['concepts_changed']} -> {perc_c_changed:.1f}%"
        )
        print(
            f"Updates Predicates \t {changes[ds]['predicates_changed']} -> {perc_p_changed:.1f}%"
        )
        print("-" * 50)


show_changes(sk_crowd, sk_final, changes)

==================== LLAVA-BENCH ====================
--------------------------------------------------
Initial Triples 	 580
Triples Discarded 	 300 -> 51.7%
Final Triples 		 516 -> 43.4% is new
Updates Concepts 	 80 -> 16.0%
Updates Predicates 	 32 -> 5.5%
--------------------------------------------------
==================== MMBENCH ====================
--------------------------------------------------
Initial Triples 	 1442
Triples Discarded 	 839 -> 58.2%
Final Triples 		 1024 -> 42.1% is new
Updates Concepts 	 77 -> 6.3%
Updates Predicates 	 110 -> 7.6%
--------------------------------------------------
==================== SEED ====================
--------------------------------------------------
Initial Triples 	 1783
Triples Discarded 	 1042 -> 58.4%
Final Triples 		 1233 -> 38.7% is new
Updates Concepts 	 101 -> 6.4%
Updates Predicates 	 126 -> 7.1%
--------------------------------------------------
==================== VQAV2 ====================
------------------------

For each dataset, many triples were marked as not relevant likely because of the scene graph generation step being task-agnostic.
While the crowdsourcing step did not fully replenish the lost relations (even if this is something we did not actively sought), the average number of concepts, relations, and their informativeness remained comparable.

In [15]:
# Compute the changes made between sk_final and sk_exp to check the same


def compute_ds_changes(final: dict[DiGraph], exp: dict[DiGraph]) -> dict:
    changes = {"concepts_changed": 0, "predicates_changed": 0}

    for idx in final:
        final_g = final[idx]
        exp_g = exp[idx]
        concepts_final = [final_g.nodes[n]["label"] for n in final_g.nodes]
        preds_final = [final_g[u][v]["label"] for u, v in final_g.edges]
        concepts_exp = [exp_g.nodes[n]["label"] for n in exp_g.nodes]
        preds_exp = [exp_g[u][v]["label"] for u, v in exp_g.edges]

        # These checks work ONLY because labels were edited and nothing else.
        for cf in concepts_final:
            if cf not in concepts_exp:
                # concept got re-labeled
                changes["concepts_changed"] += 1

        for pf in preds_final:
            if pf not in preds_exp:
                # pred got re-labeled
                changes["predicates_changed"] += 1

    return changes


def show_changes_exp(sk_final: dict, sk_exp: dict):
    for ds in DATASETS:
        changes = compute_ds_changes(sk_final["graphs"][ds], sk_exp["graphs"][ds])
        print("=" * 20, f"{ds.upper()}", "=" * 20)
        print("-" * 50)
        count_concepts = sk_exp["summaries"][ds]["count_concepts"]
        count_preds = sk_exp["summaries"][ds]["count_preds"]
        perc_c_changed = 100 * changes["concepts_changed"] / count_concepts
        perc_p_changed = 100 * changes["predicates_changed"] / count_preds
        print(
            f"Updates Concepts \t {changes['concepts_changed']} -> {perc_c_changed:.2f}% ({changes['concepts_changed']}/{count_concepts})"
        )
        print(
            f"Updates Predicates \t {changes['predicates_changed']} -> {perc_p_changed:.2f}% ({changes['predicates_changed']}/{count_preds})"
        )
        print("-" * 50)


show_changes_exp(sk_final, sk_exp)


==================== LLAVA-BENCH ====================
--------------------------------------------------
Updates Concepts 	 46 -> 7.29% (46/631)
Updates Predicates 	 48 -> 9.30% (48/516)
--------------------------------------------------
==================== MMBENCH ====================
--------------------------------------------------
Updates Concepts 	 15 -> 1.27% (15/1177)
Updates Predicates 	 54 -> 5.27% (54/1024)
--------------------------------------------------
==================== SEED ====================
--------------------------------------------------
Updates Concepts 	 86 -> 6.15% (86/1399)
Updates Predicates 	 89 -> 7.25% (89/1228)
--------------------------------------------------
==================== VQAV2 ====================
--------------------------------------------------
Updates Concepts 	 68 -> 4.46% (68/1524)
Updates Predicates 	 91 -> 7.01% (91/1298)
--------------------------------------------------


# Save stats to disk

Save statistics `sk_crowd`

In [ ]:
# out_dir = Path(".", "raw_stats", "sk_crowd")
# data_io.make_dir(out_dir)

# for ds in DATASETS:
#     print(f"Saving {ds}")

#     # Summaries
#     summary_file = out_dir.joinpath(ds, "summary.json")
#     data_io.make_dir(summary_file.parent)
#     data_io.save_json(sk_crowd["summaries"][ds], summary_file)
#     # Sample-level
#     sample_file = out_dir.joinpath(ds, "samples.json")
#     data_io.save_json(sk_crowd["sample_stats"][ds], sample_file)

Save statistics `sk_final`

In [ ]:
# out_dir = Path(".", "raw_stats", "sk_final")
# data_io.make_dir(out_dir)

# for ds in DATASETS:
#     print(f"Saving {ds}")

#     # Summaries
#     summary_file = out_dir.joinpath(ds, "summary.json")
#     data_io.make_dir(summary_file.parent)
#     data_io.save_json(sk_final["summaries"][ds], summary_file)
#     # Sample-level
#     sample_file = out_dir.joinpath(ds, "samples.json")
#     data_io.save_json(sk_final["sample_stats"][ds], sample_file)

Save statistics `sk_exp` (aka the data being used)

In [16]:
out_dir = Path(".", "raw_stats", "sk_exp")
data_io.make_dir(out_dir)

for ds in DATASETS:
    print(f"Saving {ds}")

    # Summaries
    summary_file = out_dir.joinpath(ds, "summary.json")
    data_io.make_dir(summary_file.parent)
    data_io.save_json(sk_exp["summaries"][ds], summary_file)
    # Sample-level
    sample_file = out_dir.joinpath(ds, "samples.json")
    data_io.save_json(sk_exp["sample_stats"][ds], sample_file)

Saving llava-bench
Saving mmbench
Saving seed
Saving vqav2


# Plot statistics
These are only the counts, here I directly use the data corresponding to `sk_final` since `sk_exp` only includes re-labeling and not additions/deletions.

In [16]:
# Create output dir for charts
charts_dir = Path(".", "stats_charts")
data_io.make_dir(charts_dir)

[Errno 17] File exists: 'stats_charts'


In [18]:
# Doing one with only the relations
vis.plot_sk_rels(all_count_preds, charts_dir.joinpath("box_sk.pdf"))

['llava-bench', 'mmbench', 'seed', 'vqav2']


/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["OG", "VAL", "ANN"])


In [19]:
# Doing one with both relations and concepts
vis.plot_sk(all_count_concepts, all_count_preds, charts_dir.joinpath("box_sk_full.pdf"))

/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:146: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", "", "Relations", ""])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:146: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", "", "Relations", ""])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:146: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", "", "Relations", ""])
/Users/lorenzocorti/Developer/mllm_diagnosis/utils/vis.py:146: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["", "Concepts", "", ""

In [17]:
def parse_sk_rels(rels: list[dict]) -> list[list]:
    rels_small = []
    for r in rels:
        rels_small.append(
            [
                r["from_concept"]["bb_label"]["bb_label_text"],
                r["rel_label"]["rel_label_text"],
                r["to_concept"]["bb_label"]["bb_label_text"],
            ]
        )
    return rels_small

In [18]:
sk_final = {}
sk_exp = {}

for ds in DATASETS:
    sk_hdl.set_curr_ds(ds)
    if ds not in sk_final:
        sk_final[ds] = {}
    if ds not in sk_exp:
        sk_exp[ds] = {}

    for ds_class in sk_hdl.get_classes():
        sk_hdl.set_curr_class(ds_class)
        # Load sk_final
        raw_data_path = Path(base_dir, sk_hdl.get_sk_final_path())
        raw_data = data_io.load_jsonl(raw_data_path)
        for sk_data in raw_data:
            curr_id = str(sk_data["question_id"])
            # Add entire entry
            sk_final[ds][curr_id] = sk_data
            sk_final[ds][curr_id]["relations"] = parse_sk_rels(sk_data["relations"])
            # Remove useless fields
            sk_final[ds][curr_id].pop("question_id")
            sk_final[ds][curr_id].pop("img_path")

        # Load sk_exp
        raw_data_path = Path(base_dir, sk_hdl.get_sk_exp_path())
        raw_data = data_io.load_jsonl(raw_data_path)
        for sk_data in raw_data:
            curr_id = str(sk_data["question_id"])
            # Add entire entry
            sk_exp[ds][curr_id] = sk_data
            sk_exp[ds][curr_id]["relations"] = parse_sk_rels(sk_data["relations"])
            # Remove question_id
            sk_exp[ds][curr_id].pop("question_id")
            sk_exp[ds][curr_id].pop("img_path")

In [19]:
"""
Do the checking
- Iteration over the sk_exp
- If it is absent from sk_final -> increase count
- Otherwise, go to the next sk_exp available
"""

for ds, samples in sk_exp.items():
    n_edits = 0
    n_final_rels = 0
    for q_idx, sk_data in samples.items():
        # Update to make counts correct
        n_final_rels += len(sk_final[ds][q_idx]["relations"])
        for rel in sk_data["relations"]:
            if rel not in sk_final[ds][q_idx]["relations"]:
                n_edits += 1
    print(f"{ds.upper()}: {n_edits} edits -> {100 * n_edits / n_final_rels:.1f}%")

LLAVA-BENCH: 60 edits -> 11.0%
MMBENCH: 64 edits -> 5.9%
SEED: 109 edits -> 8.4%
VQAV2: 108 edits -> 7.8%


In [20]:
# Get the final number of SKs obtained
for ds, samples in sk_exp.items():
    n_sk = 0
    for sk_data in samples.values():
        n_sk += len(sk_data["relations"])

    print(f"{ds.upper()}: {n_sk} behaviours.")

LLAVA-BENCH: 546 behaviours.
MMBENCH: 1094 behaviours.
SEED: 1301 behaviours.
VQAV2: 1391 behaviours.


Median and IQR

In [ ]:
import numpy as np

for ds_key, ds_data in sk_exp.items():
    sizes = [len(elem["relations"]) for elem in ds_data.values()]
    median = np.median(sizes).item()
    q1 = np.quantile(sizes, 0.25).item()
    q3 = np.quantile(sizes, 0.75).item()
    print(f"{ds_key.upper()}: {median} ({q1}, {q3})")

LLAVA-BENCH: 9.0 (5.75, 11.0)
MMBENCH: 7.0 (5.0, 10.5)
SEED: 9.0 (5.0, 12.0)
VQAV2: 8.5 (6.0, 12.0)
